<a href="https://colab.research.google.com/github/WaelRabah/TT_LLM/blob/main/TT_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TT_LLM: Tensor-Train Compression for LLM Linear Layers

This notebook compresses the linear layers of a HuggingFace causal LM using
**Tensor-Train (TT) decomposition** with two faithful weight initialisations
(`svd` and `tt_svd`) and one ablation (`random`).

All reusable code lives in the `tt_llm` package (`tt_llm/layers.py`, 
`tt_llm/decompositions.py`, `tt_llm/compress.py`); this notebook only handles
Colab interaction: installs, HF login, model loading, inference and comparison.

## Section 1: Environment Setup

Install the `tt_llm` package directly from the GitHub repo (works in Colab) and
the libraries required to load the model.

In [30]:
%pip install -q transformers accelerate huggingface_hub

# The TT_LLM repo is private; authenticate the install with a GitHub token
# stored in Colab Secrets (key: GITHUB_TOKEN). Fall back to a public clone
# if the token is unavailable.
try:
    from google.colab import userdata
    _gh_token = userdata.get('GITHUB_TOKEN')
    _install_url = f"git+https://{_gh_token}@github.com/WaelRabah/TT_LLM.git"
except Exception:
    _gh_token = None
    _install_url = "git+https://github.com/WaelRabah/TT_LLM.git"

%pip install -q "{_install_url}"

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## Section 2: HuggingFace Login
Retrieve the HF token from Colab Secrets (optional; only needed for gated
models).

In [31]:
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    from huggingface_hub import login
    login(token=hf_token)
except Exception as e:
    print("HF_TOKEN not found in Secrets. Set it for gated models or use a public model.")

HF_TOKEN not found in Secrets. Set it for gated models or use a public model.


## Section 3: Load the Model and Tokenizer

We use `HuggingFaceTB/SmolLM2-360M` (a small open model). The same flow works
for other `AutoModelForCausalLM` checkpoints.

In [32]:
import torch
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM

if 'model' in globals():
    del model
torch.cuda.empty_cache()
gc.collect()

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Target execution device: {device}")

model_id = "HuggingFaceTB/SmolLM2-360M"
tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16 if device == "cuda" else torch.float32,
    device_map=device,
)
print(f"Model and tokenizer loaded! Model is on: {model.device}")

Target execution device: cpu


Loading weights: 100%|██████████| 290/290 [00:01<00:00, 278.32it/s]

Model and tokenizer loaded! Model is on: cpu


## Section 4: Baseline Inference

Run a prompt on the uncompressed model to verify it produces coherent text.

In [33]:
prompt = "Explain the concept of quantum computing in one simple sentence."

def run_inference(model, prompt, **gen_kwargs):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=50, do_sample=True,
                                  temperature=0.7, top_p=0.9, **gen_kwargs)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print("--- Baseline Inference (uncompressed) ---")
print("Prompt:", prompt)
print("Response:", run_inference(model, prompt))

--- Baseline Inference (uncompressed) ---
Prompt: Explain the concept of quantum computing in one simple sentence.
Response: Explain the concept of quantum computing in one simple sentence.

What is the difference between quantum computing and classical computing?

How is quantum computing different from classical computing?

What is the difference between quantum computing and classical computing?

What are the advantages of quantum computing over classical computing?


## Section 5: Uniform Compression (SVD / TT-SVD / Random)

Recursively swap every `nn.Linear` in the model with a TT-decomposed layer.
- `init_method='svd'`: global best-rank SVD approximation of each weight
  matrix, embedded as TT cores.
- `init_method='tt_svd'`: canonical TT-SVD with per-unfolding truncation.
- `init_method='random'`: random TT cores (ablation; reproduces the original
  notebook's broken behaviour).

`compression_pct=10` means cut parameters by 10% (keep 90%).

In [34]:
from tt_llm import compress_model_inplace

print("--- Compressing with SVD init (10% reduction) ---")
compress_model_inplace(model, compression_pct=10, layer_type='linear', init_method='svd')
print("\nResponse after SVD-init compression:")
print(run_inference(model, prompt))

--- Compressing with SVD init (10% reduction) ---


KeyboardInterrupt: 

## Section 6: TT-SVD Init — Reload and Recompress

Reload the fresh model (TT-SVD is destructive; for an apples-to-apples
comparison we start from the original weights each time).

In [23]:
del model
torch.cuda.empty_cache()
gc.collect()

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16 if device == "cuda" else torch.float32,
    device_map=device,
)

print("--- Compressing with TT-SVD init (10% reduction) ---")
compress_model_inplace(model, compression_pct=10, layer_type='linear', init_method='tt_svd')
print("\nResponse after TT-SVD-init compression:")
print(run_inference(model, prompt))

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 435.00it/s]


--- Compressing with TT-SVD init (10% reduction) ---
Modified 'lm_head': Linear -> TensorLinear [tt_svd] | Comp: 1.57x
Modified 'model.layers.0.self_attn.q_proj': Linear -> TensorLinear [tt_svd] | Comp: 1.55x
Modified 'model.layers.0.self_attn.k_proj': Linear -> TensorLinear [tt_svd] | Comp: 1.59x
Modified 'model.layers.0.self_attn.v_proj': Linear -> TensorLinear [tt_svd] | Comp: 1.59x
Modified 'model.layers.0.self_attn.o_proj': Linear -> TensorLinear [tt_svd] | Comp: 1.55x
Modified 'model.layers.0.mlp.gate_proj': Linear -> TensorLinear [tt_svd] | Comp: 1.61x
Modified 'model.layers.0.mlp.up_proj': Linear -> TensorLinear [tt_svd] | Comp: 1.61x
Modified 'model.layers.0.mlp.down_proj': Linear -> TensorLinear [tt_svd] | Comp: 1.61x
Modified 'model.layers.1.self_attn.q_proj': Linear -> TensorLinear [tt_svd] | Comp: 1.55x
Modified 'model.layers.1.self_attn.k_proj': Linear -> TensorLinear [tt_svd] | Comp: 1.59x
Modified 'model.layers.1.self_attn.v_proj': Linear -> TensorLinear [tt_svd] | Comp

## Section 7: Targeted (Activation-Aware) Compression

Instead of compressing every layer by the same amount, we:
1. Run a set of calibration prompts through the model with forward hooks on
   every `nn.Linear`.
2. Score each layer's *importance* = Frobenius norm of its input activations.
3. Allocate the global parameter budget proportional to importance —
   important layers keep more parameters (less compression), less-important
   layers are compressed more aggressively.

This should preserve output quality better than uniform compression at the
same overall parameter budget.

In [35]:
from tt_llm import compress_model_targeted

del model
torch.cuda.empty_cache(); gc.collect()
model = AutoModelForCausalLM.from_pretrained(
    model_id, dtype=torch.bfloat16 if device == "cuda" else torch.float32, device_map=device,
)

calibration_prompts = [
    "Explain the concept of quantum computing in one simple sentence.",
    "Write a short story about a robot learning to paint.",
    "What are the main causes of climate change?",
    "Describe the process of photosynthesis.",
    "How does a transformer neural network work?",
]

print("--- Targeted compression (10% reduction, SVD init) ---")
compress_model_targeted(
    model, tokenizer, calibration_prompts,
    compression_pct=10, importance_cutoff=0.1, layer_type='linear', init_method='svd',
    max_length=128, device=device
)
print("\nResponse after targeted compression:")
print(run_inference(model, prompt))

Loading weights: 100%|██████████| 290/290 [00:01<00:00, 202.16it/s]

--- Targeted compression (10% reduction, SVD init) ---


TypeError: compress_model_targeted() got an unexpected keyword argument 'importance_cutoff'

## Section 8: Random-Init Ablation

The original notebook's behaviour: TT cores are random, so the compressed
model produces garbage. We keep it for comparison.

In [25]:
del model
torch.cuda.empty_cache()
gc.collect()

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16 if device == "cuda" else torch.float32,
    device_map=device,
)

print("--- Compressing with RANDOM init (ablation) ---")
compress_model_inplace(model, compression_pct=10, layer_type='linear', init_method='random')
print("\nResponse after random-init compression (expect garbage):")
print(run_inference(model, prompt))

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 328.62it/s]


--- Compressing with RANDOM init (ablation) ---
Modified 'lm_head': Linear -> TensorLinear [random] | Comp: 1.11x
Modified 'model.layers.0.self_attn.q_proj': Linear -> TensorLinear [random] | Comp: 1.11x
Modified 'model.layers.0.self_attn.k_proj': Linear -> TensorLinear [random] | Comp: 1.11x
Modified 'model.layers.0.self_attn.v_proj': Linear -> TensorLinear [random] | Comp: 1.11x
Modified 'model.layers.0.self_attn.o_proj': Linear -> TensorLinear [random] | Comp: 1.11x
Modified 'model.layers.0.mlp.gate_proj': Linear -> TensorLinear [random] | Comp: 1.11x
Modified 'model.layers.0.mlp.up_proj': Linear -> TensorLinear [random] | Comp: 1.11x
Modified 'model.layers.0.mlp.down_proj': Linear -> TensorLinear [random] | Comp: 1.11x
Modified 'model.layers.1.self_attn.q_proj': Linear -> TensorLinear [random] | Comp: 1.11x
Modified 'model.layers.1.self_attn.k_proj': Linear -> TensorLinear [random] | Comp: 1.11x
Modified 'model.layers.1.self_attn.v_proj': Linear -> TensorLinear [random] | Comp: 1.1

KeyboardInterrupt: 